In [1]:
import numpy as np
import pandas as pd
import altair as alt

movies_credits = pd.read_csv('movies_and_credits.csv')
movies_credits = movies_credits.drop(['keywords', 'tagline', 'spoken_languages', 'runtime', 'movie_id'], axis = 1)
movies_credits.head()



,title,cast,crew,budget,genres,original_language,popularity,production_companies,production_countries,release_date,revenue,vote_average,vote_count
0,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",12/10/2009,2.787965e+09,7.2,11800.0
1,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",en,139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",5/19/2007,9.610000e+08,6.9,4500.0
2,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",10/26/2015,8.806746e+08,6.3,4466.0
3,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",en,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",7/16/2012,1.084939e+09,7.6,9106.0
4,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",en,43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",3/7/2012,2.841391e+08,6.1,2124.0


In [2]:
import altair as alt
import pandas as pd

# Load data
movies_credits = pd.read_csv('movies_and_credits.csv')

# Convert release_date to datetime
movies_credits['release_date'] = pd.to_datetime(
    movies_credits['release_date'], 
    format='%m/%d/%Y', 
    errors='coerce'
)

# Drop rows with invalid dates
movies_credits = movies_credits.dropna(subset=['release_date'])

# Create year-month column
movies_credits['year_month'] = movies_credits['release_date'].dt.to_period('M').dt.to_timestamp()

# Group by year-month and count movies
monthly_data = movies_credits.groupby('year_month').size().reset_index(name='count')

# Get year range for sliders
min_year = int(movies_credits['release_date'].dt.year.min())
max_year = int(movies_credits['release_date'].dt.year.max())

# Create year sliders
start_year = alt.param(
    name='start_year',
    value=min_year,
    bind=alt.binding_range(min=min_year, max=max_year-1, step=1, name='Start Year: ')
)

end_year = alt.param(
    name='end_year',
    value=max_year,
    bind=alt.binding_range(min=min_year+1, max=max_year, step=1, name='End Year: ')
)

# Create the chart with filtering
chart = alt.Chart(monthly_data).transform_calculate(
    year='year(datum.year_month)'
).transform_filter(
    (alt.datum.year >= start_year) & (alt.datum.year <= end_year)
).mark_line(
    point=True,
    color='steelblue',
    size=3
).encode(
    x=alt.X('year_month:T', 
            title='Date (Year-Month)',
            axis=alt.Axis(format='%Y-%m', labelAngle=-45)),
    y=alt.Y('count:Q', 
            title='Number of Movies Released',
            scale=alt.Scale(zero=True)),
    tooltip=[
        alt.Tooltip('year_month:T', title='Month', format='%B %Y'),
        alt.Tooltip('count:Q', title='Movies Released')
    ]
).properties(
    width=900,
    height=400,
    title='Movie Release Patterns by Year and Month - Interactive Time Scroller'
).add_params(
    start_year,
    end_year
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='middle'
)

chart

alt.Chart(...)